# Open PII Masking 500k – Benchmark

This notebook loads the `ai4privacy/open-pii-masking-500k-ai4privacy` dataset from Hugging Face and shows how to work with it using `pandas`.

Dataset card: `https://huggingface.co/datasets/ai4privacy/open-pii-masking-500k-ai4privacy`


In [1]:
import pandas as pd
from datasets import load_dataset

/home/gregoire/silex/privacy-service/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds = load_dataset("ai4privacy/open-pii-masking-500k-ai4privacy")

ds

DatasetDict({
    train: Dataset({
        features: ['source_text', 'masked_text', 'privacy_mask', 'split', 'uid', 'language', 'region', 'script', 'mbert_tokens', 'mbert_token_classes'],
        num_rows: 464150
    })
    validation: Dataset({
        features: ['source_text', 'masked_text', 'privacy_mask', 'split', 'uid', 'language', 'region', 'script', 'mbert_tokens', 'mbert_token_classes'],
        num_rows: 116077
    })
})

In [3]:
test_df = ds["validation"].to_pandas()
french_test_df = test_df[test_df["language"] == "fr"]
french_test_df.head()

,source_text,masked_text,privacy_mask,split,uid,language,region,script,mbert_tokens,mbert_token_classes
0,Ma mère Astrit Nani Kofi est née à Ruswil en f...,Ma mère [GIVENNAME_1] [SURNAME_1] est née à [C...,"[{'label': 'GIVENNAME', 'start': 8, 'end': 19,...",validation,5706814,fr,CH,Latn,"[Ma, mère, As, ##tri, ##t, Nan, ##i, Ko, ##fi,...","[O, O, B-GIVENNAME, I-GIVENNAME, I-GIVENNAME, ..."
7,21:00 +4026945 8718 : 'Je souhaite vous contac...,[TIME_1] [TELEPHONENUM_1] : 'Je souhaite vous ...,"[{'label': 'TIME', 'start': 0, 'end': 5, 'valu...",validation,5258182,fr,FR,Latn,"[21, :, 00, +, 402, ##69, ##45, 871, ##8, :, '...","[B-TIME, I-TIME, I-TIME, B-TELEPHONENUM, I-TEL..."
10,Lesушки Noris peuvent être atteints à rocalça@...,Lesушки [GIVENNAME_1] peuvent être atteints à ...,"[{'label': 'GIVENNAME', 'start': 8, 'end': 13,...",validation,5258368,fr,CH,Latn,"[Les, ##ушки, Nor, ##is, peuvent, être, attein...","[O, O, B-GIVENNAME, I-GIVENNAME, O, O, O, O, O..."
25,Nous remercions Dzevad Gutwirth pour son souti...,Nous remercions [GIVENNAME_1] [SURNAME_1] pour...,"[{'label': 'GIVENNAME', 'start': 16, 'end': 22...",validation,5260642,fr,CA,Latn,"[Nous, re, ##mer, ##cions, D, ##ze, ##vad, Gut...","[O, O, O, O, B-GIVENNAME, I-GIVENNAME, I-GIVEN..."
31,La poème postcoloniale est intitulée : Mtre et...,La poème postcoloniale est intitulée : [TITLE_...,"[{'label': 'TITLE', 'start': 39, 'end': 43, 'v...",validation,5261762,fr,CH,Latn,"[La, poème, post, ##colo, ##nial, ##e, est, in...","[O, O, O, O, O, O, O, O, O, B-TITLE, I-TITLE, ..."


In [4]:
from privacy_service import PrivacyService

privacy_service = PrivacyService("config.example.yaml")

test_text = french_test_df["source_text"].iloc[1]
print(test_text)

print(privacy_service.detect(test_text))

Fetching 36 files: 100%|██████████| 36/36 [00:00<00:00, 189454.13it/s]


21:00 +4026945 8718 : 'Je souhaite vous contacter pour discuter de vos projets de sensibilisation au public à travers l'art du verre.'

[ai4privacy] Disclaimer 📢:
AI4Privacy is trained on the world's largest open-source privacy dataset. 
For production use, please evaluate results carefully. For assistance, contact
us at our website https://ai4privacy.com or email support@ai4privacy.com.


[ai4privacy] Loading model: ai4privacy/llama-ai4privacy-multilingual-categorical-anonymiser-openpii...
[DetectionResult(entity_type='PHONE_NUMBER', text=' +4026945', start=5, end=14, score=1.00), DetectionResult(entity_type='PHONE_NUMBER', text=' 8718', start=14, end=19, score=1.00), DetectionResult(entity_type='DATE_TIME', text='21:00', start=0, end=5, score=1.00), DetectionResult(entity_type='PHONE_NUMBER', text='+4026945 8718', start=6, end=19, score=0.40)]


In [5]:
french_test_df["detected_entities"] = french_test_df["source_text"].apply(
    lambda x: privacy_service.detect(x)
)
french_test_df.head()

,source_text,masked_text,privacy_mask,split,uid,language,region,script,mbert_tokens,mbert_token_classes,detected_entities
0,Ma mère Astrit Nani Kofi est née à Ruswil en f...,Ma mère [GIVENNAME_1] [SURNAME_1] est née à [C...,"[{'label': 'GIVENNAME', 'start': 8, 'end': 19,...",validation,5706814,fr,CH,Latn,"[Ma, mère, As, ##tri, ##t, Nan, ##i, Ko, ##fi,...","[O, O, B-GIVENNAME, I-GIVENNAME, I-GIVENNAME, ...","[DetectionResult(entity_type='PERSON', text='A..."
7,21:00 +4026945 8718 : 'Je souhaite vous contac...,[TIME_1] [TELEPHONENUM_1] : 'Je souhaite vous ...,"[{'label': 'TIME', 'start': 0, 'end': 5, 'valu...",validation,5258182,fr,FR,Latn,"[21, :, 00, +, 402, ##69, ##45, 871, ##8, :, '...","[B-TIME, I-TIME, I-TIME, B-TELEPHONENUM, I-TEL...","[DetectionResult(entity_type='PHONE_NUMBER', t..."
10,Lesушки Noris peuvent être atteints à rocalça@...,Lesушки [GIVENNAME_1] peuvent être atteints à ...,"[{'label': 'GIVENNAME', 'start': 8, 'end': 13,...",validation,5258368,fr,CH,Latn,"[Les, ##ушки, Nor, ##is, peuvent, être, attein...","[O, O, B-GIVENNAME, I-GIVENNAME, O, O, O, O, O...","[DetectionResult(entity_type='EMAIL_ADDRESS', ..."
25,Nous remercions Dzevad Gutwirth pour son souti...,Nous remercions [GIVENNAME_1] [SURNAME_1] pour...,"[{'label': 'GIVENNAME', 'start': 16, 'end': 22...",validation,5260642,fr,CA,Latn,"[Nous, re, ##mer, ##cions, D, ##ze, ##vad, Gut...","[O, O, O, O, B-GIVENNAME, I-GIVENNAME, I-GIVEN...","[DetectionResult(entity_type='PERSON', text='D..."
31,La poème postcoloniale est intitulée : Mtre et...,La poème postcoloniale est intitulée : [TITLE_...,"[{'label': 'TITLE', 'start': 39, 'end': 43, 'v...",validation,5261762,fr,CH,Latn,"[La, poème, post, ##colo, ##nial, ##e, est, in...","[O, O, O, O, O, O, O, O, O, B-TITLE, I-TITLE, ...","[DetectionResult(entity_type='PERSON', text='S..."


In [6]:
french_test_df[["source_text", "detected_entities", "privacy_mask"]].head()

,source_text,detected_entities,privacy_mask
0,Ma mère Astrit Nani Kofi est née à Ruswil en f...,"[DetectionResult(entity_type='PERSON', text='A...","[{'label': 'GIVENNAME', 'start': 8, 'end': 19,..."
7,21:00 +4026945 8718 : 'Je souhaite vous contac...,"[DetectionResult(entity_type='PHONE_NUMBER', t...","[{'label': 'TIME', 'start': 0, 'end': 5, 'valu..."
10,Lesушки Noris peuvent être atteints à rocalça@...,"[DetectionResult(entity_type='EMAIL_ADDRESS', ...","[{'label': 'GIVENNAME', 'start': 8, 'end': 13,..."
25,Nous remercions Dzevad Gutwirth pour son souti...,"[DetectionResult(entity_type='PERSON', text='D...","[{'label': 'GIVENNAME', 'start': 16, 'end': 22..."
31,La poème postcoloniale est intitulée : Mtre et...,"[DetectionResult(entity_type='PERSON', text='S...","[{'label': 'TITLE', 'start': 39, 'end': 43, 'v..."


In [7]:
from dataclasses import asdict
from IPython.display import display
import pandas as pd

# Rendre les colonnes lisibles dans les DataFrame
pd.set_option("display.max_colwidth", None)


def inspect_example(df: pd.DataFrame, idx: int) -> None:
    """Affiche proprement une ligne : texte, privacy_mask et détections.

    - df: DataFrame contenant au moins les colonnes
      `source_text`, `privacy_mask`, `detected_entities`.
    - idx: index (ou position) de la ligne à inspecter.
    """

    row = df.iloc[idx]

    print(f"Index: {idx}")
    print("\n=== Texte source ===")
    print(row["source_text"])

    print("\n=== privacy_mask (vérité terrain du dataset) ===")
    pm_df = pd.DataFrame(row["privacy_mask"])
    display(pm_df)

    print("\n=== detected_entities (PrivacyService) ===")
    det_list = [asdict(d) for d in row["detected_entities"]]
    det_df = pd.DataFrame(det_list)
    display(det_df)


# Exemple : inspecter la première ligne française
inspect_example(french_test_df, 16644)

Index: 16644

=== Texte source ===
Salut Nonso, je cherche un nouveau pot pour mon bonsaï. Pouvez-vous me recommander un magasin à Maur qui vend des pots de bonsaï ?

=== privacy_mask (vérité terrain du dataset) ===


,0
0,"{'label': 'GIVENNAME', 'start': 6, 'end': 11, 'value': 'Nonso', 'label_index': 1}"
1,"{'label': 'CITY', 'start': 96, 'end': 100, 'value': 'Maur', 'label_index': 1}"



=== detected_entities (PrivacyService) ===


,entity_type,start,end,score,text,recognizer
0,PERSON,6,11,1.000000,Nonso,EDSNLPRecognizer
1,LOCATION,96,100,1.000000,Maur,EDSNLPRecognizer
2,LOCATION,95,100,0.566858,Maur,AI4PrivacyRecognizer
3,PERSON,0,5,0.001142,Salut,AI4PrivacyRecognizer
4,GENDER,43,47,0.000423,mon,AI4PrivacyRecognizer
5,PERSON,5,12,0.000032,"Nonso,",AI4PrivacyRecognizer
6,PERSON,85,93,0.000010,magasin,AI4PrivacyRecognizer


## Evaluation

### Word-by-word classification evaluation (entity-aware)

Here, we will follow this workflow:
- Split source text by words (spaces and punctuation as separator)
- We define common entity space as a list of entities and 2 mapping:
    - ground truth (HF dataset) to common
    - detected (privacy service) to common
- Define the ground truth of the class of this word (O, for not private or the entity type mapped to the common entity space)
- Define the detected class of the word (O if not corresponding entity or the entity type detected):
    - If 0 detection capture the word then we detect O
    - If there is only one we take it
    - If several detection for the word we follow this process for this unique word:
        - eliminate a detected entity if it is contained by another
        - if conflict remains we take the highest scoring one
- Map the detected class to the common entity space
- We compute entity aware metrics for each entity type `E`:
    - TP if detected class and ground truth are both `E`
    - FN ground truth is `E` and detected is different
    - FP detected is `E` and ground truth is different

In [8]:
import re
from collections import defaultdict

from privacy_service.recognizers.entity_mapping import map_ai4privacy_to_presidio


def tokenize_with_spans(text: str) -> list[dict]:
    """Split text into tokens with character spans.

    We use a simple regex which separates words and punctuation, so every
    non-space chunk is a token. This makes it easy to align with character
    offsets from `privacy_mask` and `detected_entities`.
    """
    tokens: list[dict] = []
    for m in re.finditer(r"\w+|\S", text):
        tokens.append({"text": m.group(0), "start": m.start(), "end": m.end()})
    return tokens


def map_gt_label_to_common(label: str) -> str:
    """Map HuggingFace ground-truth label to a common entity space.

    We reuse `map_ai4privacy_to_presidio`, which maps labels like EMAIL,
    GIVENNAME, CITY, TELEPHONENUM, etc. to Presidio-like types
    (EMAIL_ADDRESS, PERSON, LOCATION, PHONE_NUMBER, ...).
    """
    return map_ai4privacy_to_presidio(label)


def map_detected_to_common(entity_type: str) -> str:
    """Map PrivacyService entity type to the same common entity space.

    PrivacyService already uses Presidio-like entity types (PERSON,
    EMAIL_ADDRESS, LOCATION, ...), so the mapping is identity here.
    """
    return entity_type


def pick_detection_for_token(token_start: int, token_end: int, detections) -> str:
    """Choose a single detected entity type for a token.

    - If 0 detections overlap the token → 'O'
    - If 1 detection overlaps → that detection's type
    - If several detections overlap:
        - eliminate a detection if its span is strictly contained in another
        - if conflict remains, keep the highest scoring one
    """
    # Collect overlapping detections
    overlapping = [
        d for d in detections if not (d.end <= token_start or d.start >= token_end)
    ]

    if not overlapping:
        return "O"

    if len(overlapping) == 1:
        return map_detected_to_common(overlapping[0].entity_type)

    # Eliminate detections that are strictly contained in another
    def is_contained(inner, outer) -> bool:
        return (
            outer.start <= inner.start
            and outer.end >= inner.end
            and (outer.start < inner.start or outer.end > inner.end)
        )

    pruned: list = []
    for d in overlapping:
        contained = any(
            is_contained(d, other) for other in overlapping if other is not d
        )
        if not contained:
            pruned.append(d)

    if not pruned:
        pruned = overlapping

    # Pick highest scoring detection
    best = max(pruned, key=lambda d: d.score)
    return map_detected_to_common(best.entity_type)


def label_tokens_for_row(row: pd.Series) -> list[dict]:
    """Produce word-level ground truth and detected labels for a row.

    Returns a list of dicts with:
    - text, start, end
    - gt_label_common: ground-truth label in common space (or 'O')
    - det_label_common: detected label in common space (or 'O')
    """
    text: str = row["source_text"]
    privacy_mask = row["privacy_mask"]
    detections = row["detected_entities"]

    tokens = tokenize_with_spans(text)

    # Pre-compute ground-truth spans in common space
    gt_spans: list[dict] = []
    for item in privacy_mask:
        gt_spans.append(
            {
                "start": item["start"],
                "end": item["end"],
                "label": map_gt_label_to_common(item["label"]),
            }
        )

    labeled_tokens: list[dict] = []
    for tok in tokens:
        t_start, t_end = tok["start"], tok["end"]

        # Ground-truth label for this token: inside any gt span → that label
        gt_label = "O"
        for gt in gt_spans:
            if t_start >= gt["start"] and t_end <= gt["end"]:
                gt_label = gt["label"]
                break

        # Detected label for this token
        det_label = pick_detection_for_token(t_start, t_end, detections)

        labeled_tokens.append(
            {
                "text": tok["text"],
                "start": t_start,
                "end": t_end,
                "gt_label": gt_label,
                "det_label": det_label,
            }
        )

    return labeled_tokens


def compute_word_level_metrics(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Compute word-level, entity-aware metrics on the whole DataFrame.

    For each token and each entity type E (in common space):
    - TP_E: gt_label == E and det_label == E
    - FN_E: gt_label == E and det_label != E
    - FP_E: det_label == E and gt_label != E
    """
    entity_counts: dict[str, dict[str, int]] = defaultdict(
        lambda: {"tp": 0, "fp": 0, "fn": 0}
    )

    # Optional: store labels per token for inspection
    all_token_rows: list[dict] = []

    for idx, row in df.iterrows():
        labeled_tokens = label_tokens_for_row(row)
        for tok in labeled_tokens:
            gt = tok["gt_label"]
            det = tok["det_label"]

            all_token_rows.append(
                {
                    "row_index": idx,
                    "text": tok["text"],
                    "start": tok["start"],
                    "end": tok["end"],
                    "gt_label": gt,
                    "det_label": det,
                }
            )

            if gt == det and gt != "O":
                entity_counts[gt]["tp"] += 1
            elif gt != "O" and det != gt:
                entity_counts[gt]["fn"] += 1
            elif det != "O" and gt != det:
                entity_counts[det]["fp"] += 1

    # Build per-entity metrics DataFrame
    rows = []
    for ent, c in entity_counts.items():
        tp, fp, fn = c["tp"], c["fp"], c["fn"]
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        rows.append(
            {
                "entity_type": ent,
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "precision": prec,
                "recall": rec,
                "f1": f1,
            }
        )

    metrics_df = pd.DataFrame(rows).sort_values("f1", ascending=False)

    tokens_df = pd.DataFrame(all_token_rows)
    return tokens_df, metrics_df


# Example usage on a subset to keep it fast (you can switch to full french_test_df)
subset = french_test_df.copy()
print("Computing word-level metrics on subset of size", len(subset))
word_labels_df, word_metrics_df = compute_word_level_metrics(subset)

print("\n=== Word-level metrics by entity type ===")
display(word_metrics_df)

print("\nSample of word-level labels:")
display(word_labels_df.head(30))

Computing word-level metrics on subset of size 22466

=== Word-level metrics by entity type ===


,entity_type,tp,fp,fn,precision,recall,f1
4,EMAIL_ADDRESS,14138,1686,196,0.893453,0.986326,0.937595
2,DATE_TIME,21656,1718,1682,0.926500,0.927929,0.927214
3,PHONE_NUMBER,19601,2161,953,0.900698,0.953634,0.926411
1,LOCATION,13913,7060,2280,0.663377,0.859198,0.748695
0,PERSON,29939,16553,3744,0.643960,0.888846,0.746841
7,US_SSN,2096,1257,171,0.625112,0.924570,0.745907
6,AGE,955,1042,171,0.478217,0.848135,0.611591
10,GENDER,747,5078,270,0.128240,0.734513,0.218357
5,IDCARDNUM,0,0,1698,0.000000,0.000000,0.000000
8,CREDIT_CARD,0,0,493,0.000000,0.000000,0.000000



Sample of word-level labels:


,row_index,text,start,end,gt_label,det_label
0,0,Ma,0,2,O,O
1,0,mère,3,7,O,O
2,0,Astrit,8,14,PERSON,PERSON
3,0,Nani,15,19,PERSON,PERSON
4,0,Kofi,20,24,PERSON,PERSON
5,0,est,25,28,O,O
6,0,née,29,32,O,O
7,0,à,33,34,O,O
8,0,Ruswil,35,41,LOCATION,LOCATION
9,0,en,42,44,O,O


### Word-by-word classification evaluation (detected-entity-type-unaware)

Here we follow the same pipeline but:
- Split source text by words (spaces and punctuation as separator)
- We define common entity space as a list of entities and 2 mapping:
    - ground truth (HF dataset) to common
    - detected (privacy service) to common
- Define the ground truth of the class of this word (O, for not private or the entity type mapped to the common entity space)
- Define the detected class of the word (O if not corresponding entity or the entity type detected):
    - If 0 detection capture the word then we detect O
    - If there is only one we take it
    - If several detection for the word we follow this process for this unique word:
        - eliminate a detected entity if it is contained by another
        - if conflict remains we take the highest scoring one
- Map the detected class to the common entity space
- We compute entity aware metrics for each entity type `E`:
    - TP if ground truth is `E` and detected class is any of the detection class (not `O`)
    - FN ground truth is `E` and no detection
    - FP detected is `E` and ground truth is `O`

In [9]:
def compute_word_level_metrics(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Compute word-level, entity-aware metrics on the whole DataFrame.

    For each token and each entity type E (in common space):
    - TP_E: ground truth is E and detected class is *any* entity (det_label != 'O')
    - FN_E: ground truth is E and no detection (det_label == 'O')
    - FP_E: detected is E and ground truth is 'O'
    """
    entity_counts: dict[str, dict[str, int]] = defaultdict(
        lambda: {"tp": 0, "fp": 0, "fn": 0}
    )

    # Optional: store labels per token for inspection
    all_token_rows: list[dict] = []

    for idx, row in df.iterrows():
        labeled_tokens = label_tokens_for_row(row)
        for tok in labeled_tokens:
            gt = tok["gt_label"]
            det = tok["det_label"]

            all_token_rows.append(
                {
                    "row_index": idx,
                    "text": tok["text"],
                    "start": tok["start"],
                    "end": tok["end"],
                    "gt_label": gt,
                    "det_label": det,
                }
            )

            # Ground truth-driven counts (TP/FN)
            if gt != "O":
                if det != "O":
                    # Any detection on a token whose GT is E counts as TP_E
                    entity_counts[gt]["tp"] += 1
                else:
                    # No detection on a token whose GT is E → FN_E
                    entity_counts[gt]["fn"] += 1

            # Detection-driven counts (FP)
            if det != "O" and gt == "O":
                # Detection of E while GT is O → FP_E
                entity_counts[det]["fp"] += 1

    # Build per-entity metrics DataFrame
    rows = []
    for ent, c in entity_counts.items():
        tp, fp, fn = c["tp"], c["fp"], c["fn"]
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        rows.append(
            {
                "entity_type": ent,
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "precision": prec,
                "recall": rec,
                "f1": f1,
            }
        )

    metrics_df = pd.DataFrame(rows).sort_values("f1", ascending=False)

    tokens_df = pd.DataFrame(all_token_rows)
    return tokens_df, metrics_df


# Example usage on a subset to keep it fast (you can switch to full french_test_df)
subset = french_test_df.copy()
print("Computing word-level metrics on subset of size", len(subset))
word_labels_df, word_metrics_df = compute_word_level_metrics(subset)

print("\n=== Word-level metrics by entity type ===")
display(word_metrics_df)

print("\nSample of word-level labels:")
display(word_labels_df.head(30))

Computing word-level metrics on subset of size 22466

=== Word-level metrics by entity type ===


,entity_type,tp,fp,fn,precision,recall,f1
13,DRIVERLICENSENUM,1883,0,66,1.000000,0.966136,0.982777
8,CREDIT_CARD,460,0,33,1.000000,0.933063,0.965373
2,DATE_TIME,22886,1718,452,0.930174,0.980632,0.954737
3,PHONE_NUMBER,20513,2161,41,0.904693,0.998005,0.949061
4,EMAIL_ADDRESS,14334,1686,0,0.894757,1.000000,0.944455
11,PASSPORTNUM,760,0,92,1.000000,0.892019,0.942928
5,IDCARDNUM,1494,0,204,1.000000,0.879859,0.936090
12,TAXNUM,2041,0,436,1.000000,0.823981,0.903497
1,LOCATION,16099,7060,94,0.695151,0.994195,0.818205
0,PERSON,33529,16553,154,0.669482,0.995428,0.800549



Sample of word-level labels:


,row_index,text,start,end,gt_label,det_label
0,0,Ma,0,2,O,O
1,0,mère,3,7,O,O
2,0,Astrit,8,14,PERSON,PERSON
3,0,Nani,15,19,PERSON,PERSON
4,0,Kofi,20,24,PERSON,PERSON
5,0,est,25,28,O,O
6,0,née,29,32,O,O
7,0,à,33,34,O,O
8,0,Ruswil,35,41,LOCATION,LOCATION
9,0,en,42,44,O,O


In [10]:
# Compute metrics for class 'O' (non-private tokens)
print("\n=== Metrics for class 'O' (non-private) ===")
if not word_labels_df.empty:
    tp_o = (
        (word_labels_df["gt_label"] == "O") & (word_labels_df["det_label"] == "O")
    ).sum()
    fn_o = (
        (word_labels_df["gt_label"] == "O") & (word_labels_df["det_label"] != "O")
    ).sum()
    fp_o = (
        (word_labels_df["gt_label"] != "O") & (word_labels_df["det_label"] == "O")
    ).sum()

    prec_o = tp_o / (tp_o + fp_o) if (tp_o + fp_o) > 0 else 0.0
    rec_o = tp_o / (tp_o + fn_o) if (tp_o + fn_o) > 0 else 0.0
    f1_o = 2 * prec_o * rec_o / (prec_o + rec_o) if (prec_o + rec_o) > 0 else 0.0

    print(f"TP_O: {tp_o}, FP_O: {fp_o}, FN_O: {fn_o}")
    print(f"Precision_O: {prec_o:.4f}")
    print(f"Recall_O: {rec_o:.4f}")
    print(f"F1_O: {f1_o:.4f}")
else:
    print("word_labels_df is empty; cannot compute metrics for class 'O'.")


=== Metrics for class 'O' (non-private) ===
TP_O: 430565, FP_O: 1583, FN_O: 37187
Precision_O: 0.9963
Recall_O: 0.9205
F1_O: 0.9569
